# Part III · Lesson 2 — Turn a PDF into a searchable collection

## Import the PDF and retrieval helpers

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
from pypdf import PdfReader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

## Choose the embedding model

In [ ]:
load_dotenv()
embeddings = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=1536)

## Open the supplied PDF

In [ ]:
# Run this notebook with its own directory as the working directory.
pdf_path = Path("../doc/Classic_Fables_Collection.pdf")
reader = PdfReader(pdf_path)
print("PDF:", pdf_path.name)
print("Pages:", len(reader.pages))

## Keep each page with its source details

In [ ]:
documents = []
for page_number, page in enumerate(reader.pages, start=1):
    text = page.extract_text() or ""
    if text.strip():
        documents.append(Document(
            page_content=text,
            metadata={"source": pdf_path.name, "page": page_number},
        ))
print(documents[3].metadata)
print(documents[3].page_content[:200])

## Split the page text into chunks

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
chunks = splitter.split_documents(documents)
for index, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = f"{pdf_path.stem}:{chunk.metadata['page']}:{index}"
chunk_ids = [chunk.metadata["chunk_id"] for chunk in chunks]
print("Pages:", len(documents), "Chunks:", len(chunks))
print(chunks[0].metadata)
print(chunks[0].page_content[:200])

## Open a persistent Chroma collection

In [ ]:
vector_store = Chroma(
    collection_name="classic_fables_cosine",
    embedding_function=embeddings,
    persist_directory="./chroma_db",
    collection_configuration={"hnsw": {"space": "cosine"}},
)

## Index once, then reuse the saved collection

In [ ]:
# This lesson uses one unchanged PDF and the same chunking setup.
if not vector_store.get(limit=1)["ids"]:
    vector_store.add_documents(documents=chunks, ids=chunk_ids)
    print("Indexed", len(chunks), "chunks.")
else:
    print("Using the saved collection; no document embeddings created.")

## Inspect one saved record and its vector dimensions

In [ ]:
saved_record = vector_store.get(
    ids=[chunk_ids[0]], include=["documents", "metadatas", "embeddings"],
)
stored_vector = saved_record["embeddings"][0]
print("ID:", saved_record["ids"][0])
print("Metadata:", saved_record["metadatas"][0])
print("Text:", saved_record["documents"][0][:120])
print("Dimensions:", len(stored_vector))
print("First 8 coordinates:", stored_vector[:8])

## Print retrieved passages with their scores

In [ ]:
def show_matches(question: str, k: int = 3):
    matches = vector_store.similarity_search_with_score(question, k=k)
    print("Question:", question)
    for document, distance in matches:
        similarity = 1 - distance
        print(f"Page {document.metadata['page']} | cosine distance {distance:.3f} | similarity {similarity:.3f}")
        print(document.page_content[:350])
        print()
    return matches

## Search for the grasshopper story

In [ ]:
grasshopper_matches = show_matches("Who helped the hungry grasshopper in winter?")

## Ask about a different story

In [ ]:
lion_matches = show_matches("How did the mouse free the lion from the hunters' net?")

## Try another query without rebuilding the index

In [ ]:
crow_matches = show_matches("How did the crow get water from the pitcher?")

## Reopen the collection after a restart

In [ ]:
reopened_store = Chroma(
    collection_name="classic_fables_cosine",
    embedding_function=embeddings,
    persist_directory="./chroma_db",
)
reopened_matches = reopened_store.similarity_search("a mouse helping a lion", k=2)
for document in reopened_matches:
    print("Page:", document.metadata["page"], document.page_content[:200])